
# 06 — ResNet + LSTM Optuna Tuning
## Purpose
Tune the ResNet + LSTM baseline with Optuna using the patch-store pipeline.

Report both global metrics and daytime-only metrics.

## Imports and settings

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import time
import random
from functools import lru_cache
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, get_worker_info

import optuna

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

### Paths & Optuna settings

In [ ]:
PROJECT_ROOT = Path("..").resolve()
DATASET_ROOT = PROJECT_ROOT / "data" / "datasets" / "manifest_v1"
GROUND_DIR   = PROJECT_ROOT / "data" / "ground_aligned"
RUNS_ROOT    = PROJECT_ROOT / "runs" / "resnet_lstm_optuna"

RUNS_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

SITE = "uniandes"
PATCH = 16
DAY_THRESHOLD = 20.0

DEBUG = False
DEBUG_TRAIN_N = 4000
DEBUG_VAL_N   = 1200
DEBUG_TEST_N  = 1200

# Optuna
N_TRIALS = 30
STUDY_NAME = f"resnet_lstm_{SITE}_P{PATCH}"

## Load metadata and manifests

In [ ]:
SITE_DIR = DATASET_ROOT / SITE
assert SITE_DIR.exists(), f"Missing dataset directory: {SITE_DIR}"

with open(SITE_DIR / "dataset_meta.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

print("Loaded dataset_meta.json:")
print(json.dumps(meta, indent=2))

MCMIPF_ROOT = Path(meta["mcmipf_root"])
PATCHES_ROOT = PROJECT_ROOT / "data" / "patches_v1" / SITE / f"P{PATCH}"
assert PATCHES_ROOT.exists(), f"Missing patches root: {PATCHES_ROOT}"

FREQ_MIN = int(meta["freq_min"])
H = int(meta["horizon_steps"])
L = int(meta["history_steps"])

GRID_SIZE = int(meta["grid_size"])
SITE_CENTER = (int(meta["site_center_pix"]["row"]), int(meta["site_center_pix"]["col"]))

train_man = pd.read_parquet(SITE_DIR / "manifest_train.parquet")
val_man   = pd.read_parquet(SITE_DIR / "manifest_val.parquet")
test_man  = pd.read_parquet(SITE_DIR / "manifest_test.parquet")

print("train:", train_man.shape)
print("val:  ", val_man.shape)
print("test: ", test_man.shape)

In [ ]:
def _hist_len(x):
    if isinstance(x, str):
        x = json.loads(x)
    elif isinstance(x, np.ndarray):
        x = x.tolist()
    return len(x)

L_manifest = int(train_man["history_ts"].map(_hist_len).mode().iloc[0])
L_meta = int(meta["history_steps"])
if L_manifest != L_meta:
    print(f"[WARN] meta L={L_meta} but manifest L={L_manifest}. Using manifest L.")
L = L_manifest

## Reproducibility

In [ ]:
SEED = int(meta.get("seed", 42))

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("SEED:", SEED)

def seed_worker(worker_id: int) -> None:
    worker_seed = (SEED + worker_id) % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

In [ ]:
if DEBUG:
    train_man = train_man.sample(n=min(DEBUG_TRAIN_N, len(train_man)), random_state=SEED).reset_index(drop=True)
    val_man   = val_man.sample(n=min(DEBUG_VAL_N, len(val_man)), random_state=SEED).reset_index(drop=True)
    test_man  = test_man.sample(n=min(DEBUG_TEST_N, len(test_man)), random_state=SEED).reset_index(drop=True)

print("DEBUG:", DEBUG)
print("train:", train_man.shape)
print("val:  ", val_man.shape)
print("test: ", test_man.shape)

## Persistence baseline

In [ ]:
ground_path = GROUND_DIR / f"ground_10min_utc_{SITE}.parquet"
assert ground_path.exists(), f"Missing ground parquet for {SITE}: {ground_path}"

ground = pd.read_parquet(ground_path)
assert "ghi" in ground.columns, "Ground dataset missing 'ghi'"
assert str(ground.index.tz) == "UTC", "Ground index must be UTC"

## Metrics

In [ ]:
def metrics_from_arrays(y_true: np.ndarray, y_pred: np.ndarray, day_threshold: float = 20.0) -> Dict[str, float]:
    y_true = y_true.astype(np.float64)
    y_pred = y_pred.astype(np.float64)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae  = float(np.mean(np.abs(y_true - y_pred)))

    day_mask = y_true >= day_threshold
    if day_mask.sum() > 0:
        rmse_day = float(np.sqrt(np.mean((y_true[day_mask] - y_pred[day_mask]) ** 2)))
        mae_day  = float(np.mean(np.abs(y_true[day_mask] - y_pred[day_mask])))
        n_day = int(day_mask.sum())
    else:
        rmse_day = float("nan")
        mae_day = float("nan")
        n_day = 0

    return {
        "n": int(len(y_true)),
        "rmse": rmse,
        "mae": mae,
        "n_day": n_day,
        "rmse_day": rmse_day,
        "mae_day": mae_day,
        "day_threshold": float(day_threshold),
    }

def skill_score(rmse_model: float, rmse_baseline: float) -> float:
    return float(1.0 - rmse_model / (rmse_baseline + 1e-12))

In [ ]:
def eval_persistence(manifest: pd.DataFrame, ground_df: pd.DataFrame, day_threshold: float = 20.0) -> Dict[str, float]:
    t_label = pd.to_datetime(manifest["t_label"], utc=True)
    y_true = manifest["y"].astype(float).to_numpy()
    y_hat = ground_df.reindex(t_label)["ghi"].to_numpy()
    return metrics_from_arrays(y_true, y_hat, day_threshold=day_threshold)

baseline_train = eval_persistence(train_man, ground, day_threshold=DAY_THRESHOLD)
baseline_val   = eval_persistence(val_man, ground, day_threshold=DAY_THRESHOLD)
baseline_test  = eval_persistence(test_man, ground, day_threshold=DAY_THRESHOLD)

print("=== Persistence baseline ===")
print("train:", baseline_train)
print("val:  ", baseline_val)
print("test: ", baseline_test)

## Target normalization

In [ ]:
y_train = train_man["y"].astype(float).to_numpy()
Y_MEAN = float(np.mean(y_train))
Y_STD  = float(np.std(y_train) + 1e-6)

def norm_y(y: float) -> float:
    return (y - Y_MEAN) / Y_STD

def denorm_y(arr: np.ndarray) -> np.ndarray:
    return arr * Y_STD + Y_MEAN

print("Target stats (train):")
print("  mean:", Y_MEAN)
print("  std :", Y_STD)
print("  percentiles:", np.percentile(y_train, [0, 50, 90, 95, 99]).tolist())

## Patch store

In [ ]:
PATCH_HOUR_CACHE_MAXSIZE = 16

def patch_path_for_timestamp(t: pd.Timestamp) -> Path:
    ymd = t.strftime("%Y%m%d")
    hh  = t.strftime("%H")
    year  = t.strftime("%Y")
    month = t.strftime("%m")
    fname = f"{ymd}_{hh}_patch.npz"
    return PATCHES_ROOT / year / month / fname

def slot_for_timestamp(t: pd.Timestamp) -> int:
    return int(t.strftime("%M")) // 10

def load_patch_npz_nocache(path_str: str) -> np.ndarray:
    p = Path(path_str)
    with np.load(p) as d:
        arr = d["patch"]  # (6,16,P,P)
    return arr

@lru_cache(maxsize=PATCH_HOUR_CACHE_MAXSIZE)
def load_patch_npz_maincache(path_str: str) -> np.ndarray:
    return load_patch_npz_nocache(path_str)

def load_patch_npz(path_str: str) -> np.ndarray:
    if get_worker_info() is None:
        return load_patch_npz_maincache(path_str)
    return load_patch_npz_nocache(path_str)

## Datasets

In [ ]:
# %%
class PatchSeqDataset(Dataset):
    """
    Returns:
      x_seq: torch.FloatTensor (L, C=16, P, P)
      y:     torch.FloatTensor scalar (normalized)
    """
    def __init__(self, manifest: pd.DataFrame):
        self.man = manifest.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.man)

    def __getitem__(self, i: int):
        row = self.man.iloc[i]
        y = norm_y(float(row["y"]))

        history_ts = row["history_ts"]
        if isinstance(history_ts, str):
            history_ts = json.loads(history_ts)
        elif isinstance(history_ts, np.ndarray):
            history_ts = history_ts.tolist()

        xs = []
        for ts_str in history_ts:
            t = pd.to_datetime(ts_str, utc=True)
            p = patch_path_for_timestamp(t)
            slot = slot_for_timestamp(t)

            if not p.exists():
                raise FileNotFoundError(f"Missing patch file: {p}")

            tensor = load_patch_npz(str(p))   # (6,16,P,P)
            frame = tensor[slot]              # (16,P,P)
            frame = np.nan_to_num(frame, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32, copy=False)
            xs.append(frame)

        x_seq = np.stack(xs, axis=0)  # (L,16,P,P)
        return torch.from_numpy(x_seq), torch.tensor(y, dtype=torch.float32)

train_ds = PatchSeqDataset(train_man)
val_ds   = PatchSeqDataset(val_man)
test_ds  = PatchSeqDataset(test_man)

print("datasets:", len(train_ds), len(val_ds), len(test_ds))

## Dataloaders

In [ ]:
def make_loader(ds: Dataset, batch_size: int, shuffle: bool, num_workers: int) -> DataLoader:
    kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=(DEVICE == "cuda"),
        worker_init_fn=seed_worker,
        generator=g,
        persistent_workers=(num_workers > 0),
    )
    if num_workers > 0:
        kwargs["prefetch_factor"] = 2
    return DataLoader(ds, **kwargs)

## Model

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)

        self.down = None
        if stride != 1 or in_ch != out_ch:
            self.down = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.down is not None:
            identity = self.down(identity)
        out = self.relu(out + identity)
        return out

class SmallResNetEncoder(nn.Module):
    def __init__(self, in_ch: int = 16, base: int = 32, emb_dim: int = 128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, base, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(base),
            nn.ReLU(inplace=True),
        )

        self.layer1 = nn.Sequential(
            BasicBlock(base, base, stride=1),
            BasicBlock(base, base, stride=1),
        )
        self.layer2 = nn.Sequential(
            BasicBlock(base, base * 2, stride=2),
            BasicBlock(base * 2, base * 2, stride=1),
        )
        self.layer3 = nn.Sequential(
            BasicBlock(base * 2, base * 4, stride=2),
            BasicBlock(base * 4, base * 4, stride=1),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.proj = nn.Linear(base * 4, emb_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.stem(x)
        h = self.layer1(h)
        h = self.layer2(h)
        h = self.layer3(h)
        h = self.pool(h).squeeze(-1).squeeze(-1)
        z = self.proj(h)
        return z

class ResNetLSTM(nn.Module):
    def __init__(self, in_ch: int = 16, base: int = 32, emb_dim: int = 128, hidden_t: int = 128, dropout: float = 0.1):
        super().__init__()
        self.encoder = SmallResNetEncoder(in_ch=in_ch, base=base, emb_dim=emb_dim)
        self.emb_norm = nn.LayerNorm(emb_dim)
        self.lstm = nn.LSTM(input_size=emb_dim, hidden_size=hidden_t, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_t, hidden_t),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_t, 1),
        )

    def forward(self, x_seq: torch.Tensor) -> torch.Tensor:
        B, Ls, C, P, P2 = x_seq.shape
        assert P == P2, "Patch must be square"

        x = x_seq.reshape(B * Ls, C, P, P)
        z = self.encoder(x)
        z = self.emb_norm(z)
        z = z.reshape(B, Ls, -1)

        out, _ = self.lstm(z)
        last = out[:, -1]
        yhat = self.head(last).squeeze(-1)
        return yhat

## Training

In [ ]:
@torch.no_grad()
def eval_model(model: nn.Module, loader: DataLoader, day_threshold: float = 20.0) -> Dict[str, float]:
    model.eval()
    ys, yhats = [], []

    for x_seq, y in loader:
        x_seq = x_seq.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        yhat = model(x_seq)

        ys.append(y.detach().cpu().numpy())
        yhats.append(yhat.detach().cpu().numpy())

    y = np.concatenate(ys)
    yhat = np.concatenate(yhats)

    y_phys = denorm_y(y)
    yhat_phys = denorm_y(yhat)

    return metrics_from_arrays(y_phys, yhat_phys, day_threshold=day_threshold)

In [ ]:
def train_one_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float,
    weight_decay: float,
    grad_clip_norm: float,
    use_amp: bool,
    epochs: int,
    patience: int,
    min_delta: float,
    day_threshold: float,
) -> Dict[str, Any]:

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
    loss_fn = nn.MSELoss()
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    train_log = []
    best_val_rmse_day = float("inf")
    best_state = None
    best_epoch = 0
    bad_epochs = 0

    t_train0 = time.time()

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        tr_losses = []

        for x_seq, y in train_loader:
            x_seq = x_seq.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                yhat = model(x_seq)
                loss = loss_fn(yhat, y)

            scaler.scale(loss).backward()

            if grad_clip_norm is not None and grad_clip_norm > 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)

            scaler.step(opt)
            scaler.update()

            tr_losses.append(loss.item())

        val_metrics = eval_model(model, val_loader, day_threshold=day_threshold)
        val_rmse = float(val_metrics["rmse"])
        val_rmse_day = float(val_metrics["rmse_day"])
        val_mae = float(val_metrics["mae"])
        val_mae_day = float(val_metrics["mae_day"])

        scheduler.step(val_rmse_day)

        epoch_out = {
            "epoch": epoch,
            "train_mse_norm": float(np.mean(tr_losses)),
            "val_rmse_phys": val_rmse,
            "val_mae_phys": val_mae,
            "val_rmse_day_phys": val_rmse_day,
            "val_mae_day_phys": val_mae_day,
            "lr": float(opt.param_groups[0]["lr"]),
            "epoch_seconds": float(time.time() - t0),
        }
        train_log.append(epoch_out)

        improved = (best_val_rmse_day - val_rmse_day) > min_delta
        if improved:
            best_val_rmse_day = val_rmse_day
            best_epoch = epoch
            bad_epochs = 0
            best_state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "best_epoch": best_epoch,
                "best_val_rmse_day": best_val_rmse_day,
            }
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    total_train_seconds = float(time.time() - t_train0)

    assert best_state is not None, "No best state captured during training."
    model.load_state_dict(best_state["model_state"])

    final_val = eval_model(model, val_loader, day_threshold=day_threshold)

    return {
        "model": model,
        "best_epoch": int(best_state["best_epoch"]),
        "best_val_rmse_day": float(best_state["best_val_rmse_day"]),
        "final_val": final_val,
        "train_log": train_log,
        "train_seconds_total": total_train_seconds,
    }

## Optuna

### Objective

In [ ]:
USE_AMP = (DEVICE == "cuda")
GRAD_CLIP_NORM = 1.0
EPOCHS = 20
PATIENCE = 6
MIN_DELTA = 0.0

In [ ]:
def objective(trial: optuna.Trial) -> float:
    base = trial.suggest_categorical("base", [16, 24, 32, 48])
    emb_dim = trial.suggest_categorical("emb_dim", [64, 128, 192])
    hidden_t = trial.suggest_categorical("hidden_t", [64, 128, 192])
    dropout = trial.suggest_categorical("dropout", [0.0, 0.1, 0.2, 0.3])

    lr = trial.suggest_float("lr", 3e-4, 3e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    train_loader = make_loader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader   = make_loader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    model = ResNetLSTM(
        in_ch=16,
        base=base,
        emb_dim=emb_dim,
        hidden_t=hidden_t,
        dropout=dropout,
    ).to(DEVICE)

    out = train_one_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        lr=lr,
        weight_decay=weight_decay,
        grad_clip_norm=GRAD_CLIP_NORM,
        use_amp=USE_AMP,
        epochs=EPOCHS,
        patience=PATIENCE,
        min_delta=MIN_DELTA,
        day_threshold=DAY_THRESHOLD,
    )

    val_metrics = out["final_val"]
    n_params = sum(p.numel() for p in model.parameters())

    trial.set_user_attr("best_epoch", out["best_epoch"])
    trial.set_user_attr("val_rmse", float(val_metrics["rmse"]))
    trial.set_user_attr("val_mae", float(val_metrics["mae"]))
    trial.set_user_attr("val_rmse_day", float(val_metrics["rmse_day"]))
    trial.set_user_attr("val_mae_day", float(val_metrics["mae_day"]))
    trial.set_user_attr("train_seconds_total", float(out["train_seconds_total"]))
    trial.set_user_attr("n_params", int(n_params))

    return float(val_metrics["rmse_day"])

## Run

### Optuna

In [ ]:
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

In [ ]:
print("Best trial:")
print("  value (val_rmse_day):", study.best_trial.value)
print("  params:")
for k, v in study.best_trial.params.items():
    print(f"    {k}: {v}")

## Summary

### Trials

In [ ]:
rows = []
for t in study.trials:
    row = {
        "number": t.number,
        "state": str(t.state),
        "objective_val_rmse_day": t.value,
    }
    row.update(t.params)
    row.update(t.user_attrs)
    rows.append(row)

trials_df = pd.DataFrame(rows).sort_values("objective_val_rmse_day", ascending=True)
trials_df.head(10)

## Retrain

In [ ]:
best_params = study.best_trial.params
best_params

In [ ]:
FINAL_EPOCHS = 30
FINAL_PATIENCE = 8

best_batch_size = best_params["batch_size"]

train_loader = make_loader(train_ds, batch_size=best_batch_size, shuffle=True, num_workers=4)
val_loader   = make_loader(val_ds, batch_size=best_batch_size, shuffle=False, num_workers=0)
test_loader  = make_loader(test_ds, batch_size=best_batch_size, shuffle=False, num_workers=0)

best_model = ResNetLSTM(
    in_ch=16,
    base=best_params["base"],
    emb_dim=best_params["emb_dim"],
    hidden_t=best_params["hidden_t"],
    dropout=best_params["dropout"],
).to(DEVICE)

out = train_one_model(
    model=best_model,
    train_loader=train_loader,
    val_loader=val_loader,
    lr=best_params["lr"],
    weight_decay=best_params["weight_decay"],
    grad_clip_norm=GRAD_CLIP_NORM,
    use_amp=USE_AMP,
    epochs=FINAL_EPOCHS,
    patience=FINAL_PATIENCE,
    min_delta=MIN_DELTA,
    day_threshold=DAY_THRESHOLD,
)

final_val = out["final_val"]
final_test = eval_model(best_model, test_loader, day_threshold=DAY_THRESHOLD)

final_val["skill_vs_persistence"] = skill_score(final_val["rmse"], baseline_val["rmse"])
final_val["skill_day_vs_persistence"] = skill_score(final_val["rmse_day"], baseline_val["rmse_day"])

final_test["skill_vs_persistence"] = skill_score(final_test["rmse"], baseline_test["rmse"])
final_test["skill_day_vs_persistence"] = skill_score(final_test["rmse_day"], baseline_test["rmse_day"])

print("Best tuned ResNet val:", final_val)
print("Best tuned ResNet test:", final_test)

## Save outputs

In [ ]:
run_name = f"{SITE}_H{H}_L{L}_P{PATCH}_seed{SEED}_{pd.Timestamp.now('UTC').strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = RUNS_ROOT / run_name
RUN_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    "run_name": run_name,
    "study_name": STUDY_NAME,
    "site": SITE,
    "device": DEVICE,
    "seed": SEED,
    "day_threshold": DAY_THRESHOLD,
    "data_paths": {
        "site_dir": str(SITE_DIR),
        "mcmipf_root": str(MCMIPF_ROOT),
        "patches_root": str(PATCHES_ROOT),
        "ground_path": str(ground_path),
        "run_dir": str(RUN_DIR),
    },
    "temporal": {
        "freq_min": int(FREQ_MIN),
        "history_steps_L": int(L),
        "horizon_steps_H": int(H),
        "history_hours": float(L * FREQ_MIN / 60.0),
        "horizon_hours": float(H * FREQ_MIN / 60.0),
    },
    "spatial": {
        "grid_size": int(GRID_SIZE),
        "patch": int(PATCH),
        "site_center_rc": {"row": int(SITE_CENTER[0]), "col": int(SITE_CENTER[1])},
        "channels": 16,
    },
    "target_norm": {
        "y_mean_train": float(Y_MEAN),
        "y_std_train": float(Y_STD),
    },
    "baselines": {
        "persistence_train": baseline_train,
        "persistence_val": baseline_val,
        "persistence_test": baseline_test,
    },
    "optuna": {
        "n_trials": int(N_TRIALS),
        "best_value_val_rmse_day": float(study.best_trial.value),
        "best_params": study.best_trial.params,
    },
    "best_model": {
        "arch": "SmallResNetEncoder + LayerNorm + LSTM + MLP head",
        "best_epoch": int(out["best_epoch"]),
        "train_seconds_total": float(out["train_seconds_total"]),
        "final_val": final_val,
        "final_test": final_test,
        "n_params": int(sum(p.numel() for p in best_model.parameters())),
    },
}

with open(RUN_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

trials_df.to_csv(RUN_DIR / "trials.csv", index=False)

print("Saved summary to:", RUN_DIR / "summary.json")
print("Saved trials to :", RUN_DIR / "trials.csv")